# 101 - Getting Started with ThetaDataDx

This notebook walks through the basics of the `thetadatadx` Python SDK:
installing the package, authenticating, and fetching stock and option data.

**Prerequisites:** A ThetaData account with an active subscription.

## 1. Installation

Install the SDK with pandas support. The package ships a pre-compiled
native extension - no JVM required.

In [ ]:
# Uncomment to install (run once)
# !pip install thetadatadx[pandas]

## 2. Connect with Credentials

Create a `Credentials` object with your ThetaData email and password,
then connect a `Client` to the production servers. The client
opens its data connection at construction.

In [ ]:
from thetadatadx import Credentials, Config, Client

# Option A: inline credentials
# creds = Credentials("your-email@example.com", "your-password")

# Option B: load from a file (line 1 = email, line 2 = password)
creds = Credentials.from_file("creds.txt")

config = Config.production()
client = Client(creds, config)
print("Connected:", client)

## 3. List Stock Symbols

Retrieve the full universe of available stock tickers. List endpoints
return a typed `StringList`; iterate it directly or call `.to_list()`
for a plain Python list.

In [ ]:
symbols = client.market_data.stock_list_symbols().to_list()
print(f"Total symbols available: {len(symbols)}")
print(f"First 20: {symbols[:20]}")

## 4. Fetch AAPL End-of-Day Data as a DataFrame

`stock_history_eod` returns a typed `EodTickList`. Chain `.to_pandas()`
(or `.to_polars()` / `.to_arrow()` / `.to_list()`) on the result for the
representation you need. Each row carries OHLCV plus the closing NBBO and
two time columns: `created_ms_of_day` (when the EOD report was generated)
and `last_trade_ms_of_day` (the last trade of the session), both in
milliseconds since midnight ET.

In [ ]:
import pandas as pd

df_eod = client.market_data.stock_history_eod("AAPL", "20240101", "20240401").to_pandas()

print(f"Rows: {len(df_eod)}")
df_eod[["date", "open", "high", "low", "close", "volume",
        "created_ms_of_day", "last_trade_ms_of_day"]].head(10)

## 5. Fetch 1-Minute OHLC Bars

Intraday bars for a single trading day. The `interval` argument is the
bar width as a duration string: `"1m"` = one minute, `"1s"` = one second,
`"5m"` = five minutes. The bar timestamp `ms_of_day` is the bar's opening
time in milliseconds since midnight ET (34_200_000 = 09:30).

In [ ]:
df_ohlc = client.market_data.stock_history_ohlc("AAPL", "20240315", interval="1m").to_pandas()

print(f"1-min bars on 2024-03-15: {len(df_ohlc)}")
df_ohlc[["ms_of_day", "open", "high", "low", "close", "volume"]].head(10)

## 6. List SPY Option Expirations and Strikes

Options data starts with listing available expirations, then the strikes
for a given expiration. Expirations come back as ISO date strings
(`"2024-03-15"`) and strikes as dollar-value strings (`"540"`).

In [ ]:
# All available expirations for SPY (ISO date strings)
expirations = client.market_data.option_list_expirations("SPY").to_list()
print(f"Total expirations: {len(expirations)}")

# Keep only expirations in the future
import datetime
today = datetime.date.today().isoformat()
future = [e for e in expirations if e > today]
print(f"Next 5 upcoming: {future[:5]}")

# Strikes for the nearest upcoming expiration (dollar strings)
nearest_exp = future[0]
strikes = client.market_data.option_list_strikes("SPY", nearest_exp).to_list()
print(f"\nStrikes for {nearest_exp}: {len(strikes)} total")
print(f"Range: {strikes[0]} - {strikes[-1]}")
mid = len(strikes) // 2
print(f"Sample near the middle: {strikes[mid - 3:mid + 3]}")

---

**Next:** [102 - Historical Data Analysis](102_historical_analysis.ipynb)